# YOLOv4-Tiny 無腦訓練 Notebook

## 資料夾結構（自動建立）
```
MyDrive/
└── yolov4_tiny_training/
    ├── dataset/      ← 把圖片和 .txt 標註放這裡
    ├── backup/       ← 訓練過程中自動儲存 weights
    └── logs/         ← training log 與 loss 曲線
```

## YOLO 標註格式
每張圖片對應一個同名 `.txt`，每行一個物件：
```
<class_id> <x_center> <y_center> <width> <height>
```
（所有值皆為相對比例，範圍 0~1）

## 使用步驟
1. 修改下方「使用者設定區」
2. 執行 **Cell 1**（設定）→ **Cell 2**（掛載 Drive）
3. 把圖片與 `.txt` 上傳到 Drive 的 `yolov4_tiny_training/dataset/`
4. 確認上傳完畢後，繼續執行剩餘 Cells（或直接 Runtime → Run all）

In [ ]:
# ============================================================
#  使用者設定區 ── 只需修改這個 Cell，其餘全部自動執行
# ============================================================

# ── 類別設定 ─────────────────────────────────────────────────
CLASS_NAMES = [
    '0', '1', '2', '3', '4',
    '5', '6', '7', '8', '9',
    'add', 'sub', 'mul', 'div'
]

# ── 資料分割比例（三者相加必須 = 1.0）────────────────────────
TRAIN_RATIO = 0.7   # 訓練集
VAL_RATIO   = 0.2   # 驗證集
TEST_RATIO  = 0.1   # 測試集（設 0 則不建立 test set）

# ── 訓練超參數 ────────────────────────────────────────────────
BATCH_SIZE   = 64
SUBDIVISIONS = 16
# None → 自動計算 max(6000, classes * 2000)
MAX_BATCHES  = None
# 輸入圖片解析度（必須為 32 的倍數）
INPUT_WIDTH  = 416
INPUT_HEIGHT = 416

# ── 隨機種子 ──────────────────────────────────────────────────
RANDOM_SEED = 42

# ============================================================
#  以下請勿修改
# ============================================================
import os

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, \
    'TRAIN_RATIO + VAL_RATIO + TEST_RATIO 必須等於 1.0'
assert INPUT_WIDTH  % 32 == 0, 'INPUT_WIDTH 必須為 32 的倍數'
assert INPUT_HEIGHT % 32 == 0, 'INPUT_HEIGHT 必須為 32 的倍數'

NUM_CLASSES = len(CLASS_NAMES)
if MAX_BATCHES is None:
    MAX_BATCHES = max(6000, NUM_CLASSES * 2000)
STEPS1  = int(MAX_BATCHES * 0.8)
STEPS2  = int(MAX_BATCHES * 0.9)
FILTERS = (NUM_CLASSES + 5) * 3

# ── 路徑定義 ──────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
PROJECT_DIR  = os.path.join(DRIVE_ROOT, 'yolov4_tiny_training')
DATASET_DIR  = os.path.join(PROJECT_DIR, 'dataset')
BACKUP_DIR   = os.path.join(PROJECT_DIR, 'backup')
LOG_DIR      = os.path.join(PROJECT_DIR, 'logs')
DARKNET_DIR  = '/content/darknet'
DATA_DIR     = os.path.join(DARKNET_DIR, 'data', 'obj')
DARKNET_BIN  = os.path.join(DARKNET_DIR, 'darknet')

print('=== 設定確認 ===')
print(f'類別數量    : {NUM_CLASSES}')
print(f'類別名稱    : {CLASS_NAMES}')
print(f'MAX_BATCHES : {MAX_BATCHES}  (steps: {STEPS1}, {STEPS2})')
print(f'FILTERS     : {FILTERS}')
print(f'解析度      : {INPUT_WIDTH} x {INPUT_HEIGHT}')
print(f'Drive 專案  : {PROJECT_DIR}')

In [ ]:
# ── Step 1：掛載 Google Drive，建立資料夾結構 ─────────────────
from google.colab import drive
import sys

drive.mount('/content/drive')

# 建立所有需要的資料夾
for d in [PROJECT_DIR, DATASET_DIR, BACKUP_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# 檢查資料集是否已有內容
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
from pathlib import Path

existing_images = [p for p in Path(DATASET_DIR).iterdir()
                   if p.suffix.lower() in IMG_EXTS]

print('\n=== Google Drive 資料夾結構 ===')
print(f'  {PROJECT_DIR}/')
print(f'  ├── dataset/   ← 圖片 + .txt 標註放這裡')
print(f'  ├── backup/    ← weights 輸出')
print(f'  └── logs/      ← training log')

if not existing_images:
    print()
    print('=' * 55)
    print('  資料夾已建立，但 dataset/ 目前是空的！')
    print()
    print('  請將圖片與對應的 .txt 標註檔上傳至：')
    print(f'  {DATASET_DIR}')
    print()
    print('  上傳完成後，再繼續執行下方 Cells。')
    print('=' * 55)
    raise SystemExit('請上傳資料後再繼續執行')
else:
    print(f'\n資料集偵測到 {len(existing_images)} 張圖片，繼續執行 ✓')

In [ ]:
# ── Step 2：安裝依賴並編譯 Darknet（GPU + OpenCV + cuDNN）──────
import subprocess, glob as _glob

print('更新套件列表...')
!apt-get update -qq 2>&1 | tail -2

print('安裝 OpenCV 開發套件（C++ headers）...')
!apt-get install -y --fix-missing libopencv-dev 2>&1 | tail -3

if not _glob.glob('/usr/include/opencv4/opencv2/opencv.hpp') and \
   not _glob.glob('/usr/include/opencv2/opencv.hpp'):
    raise RuntimeError(
        'OpenCV C++ header 安裝失敗\n'
        '請在新 Cell 手動執行：\n'
        '  !apt-get update && apt-get install -y libopencv-dev\n'
        '確認成功後再重跑本 Cell'
    )
print('OpenCV 安裝完成 ✓')

if not os.path.isdir(DARKNET_DIR):
    !git clone https://github.com/AlexeyAB/darknet {DARKNET_DIR}

%cd {DARKNET_DIR}
!sed -i 's/GPU=0/GPU=1/'               Makefile
!sed -i 's/CUDNN=0/CUDNN=1/'           Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/OPENCV=0/OPENCV=1/'         Makefile
!sed -i 's/LIBSO=0/LIBSO=1/'           Makefile

print('開始編譯 Darknet（約 1~3 分鐘）...')
# 不加 tail，保留完整輸出方便除錯
!make -j$(nproc) 2>&1

if not os.path.isfile(DARKNET_BIN):
    raise RuntimeError(
        f'Darknet 編譯失敗，找不到 {DARKNET_BIN}\n'
        '請確認 Runtime 已啟用 GPU（Runtime → Change runtime type → T4 GPU）\n'
        '如已啟用 GPU 仍失敗，請查看上方 make 輸出'
    )
print(f'Darknet 編譯完成 ✓  ({DARKNET_BIN})')

In [ ]:
# ── Step 3：掃描資料集、驗證配對、分割 train/val/test ──────────
import random, shutil

dataset_path = Path(DATASET_DIR)
all_images   = [p for p in dataset_path.iterdir() if p.suffix.lower() in IMG_EXTS]

valid_pairs, missing_labels = [], []
for img in all_images:
    if img.with_suffix('.txt').exists():
        valid_pairs.append(img)
    else:
        missing_labels.append(img.name)

if missing_labels:
    print(f'⚠ 找不到對應 .txt，已略過 {len(missing_labels)} 張：')
    for name in missing_labels[:10]:
        print(f'   {name}')
    if len(missing_labels) > 10:
        print(f'   ... 共 {len(missing_labels)} 張')

assert len(valid_pairs) >= 10, \
    f'有效圖片數量不足（{len(valid_pairs)} 張），至少需要 10 張'

# 圖片數量警告
if len(valid_pairs) < BATCH_SIZE:
    print(f'\n⚠ 警告：圖片數量（{len(valid_pairs)}）小於 BATCH_SIZE（{BATCH_SIZE}）')
    print(f'   建議回到設定區將 BATCH_SIZE 調低（例如 16 或 8）再重新執行')
if len(valid_pairs) < 50:
    print(f'\n⚠ 警告：圖片數量很少（{len(valid_pairs)} 張），建議至少每類 30 張以上')

random.seed(RANDOM_SEED)
random.shuffle(valid_pairs)

n_total = len(valid_pairs)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
if n_val < 1:
    n_val = 1
if n_train < 1:
    n_train = n_total - n_val

splits = {
    'train': valid_pairs[:n_train],
    'val'  : valid_pairs[n_train:n_train + n_val],
    'test' : valid_pairs[n_train + n_val:]
}

print(f'資料集統計')
print(f'  總計有效 : {n_total} 張')
print(f'  Train    : {len(splits["train"])} 張 ({len(splits["train"])/n_total:.1%})')
print(f'  Val      : {len(splits["val"])} 張 ({len(splits["val"])/n_total:.1%})')
print(f'  Test     : {len(splits["test"])} 張 ({len(splits["test"])/n_total:.1%})')

# 複製圖片與標註到 darknet data/obj/
os.makedirs(DATA_DIR, exist_ok=True)
print('\n複製資料到 Darknet 目錄中...')
for img_path in valid_pairs:
    shutil.copy2(str(img_path), DATA_DIR)
    shutil.copy2(str(img_path.with_suffix('.txt')), DATA_DIR)
print(f'複製完成 ✓（{n_total} 張圖 + {n_total} 個標註）')

# 產生 train.txt / val.txt / test.txt
list_dir = os.path.join(DARKNET_DIR, 'data')
for split_name, paths in splits.items():
    if not paths:
        continue
    list_path = os.path.join(list_dir, f'{split_name}.txt')
    with open(list_path, 'w') as f:
        for p in paths:
            f.write(f'{DATA_DIR}/{p.name}\n')
    print(f'{split_name}.txt 已建立 ✓')

# 驗證第一張圖片路徑真的存在
first_img = f'{DATA_DIR}/{splits["train"][0].name}'
assert os.path.isfile(first_img), f'路徑驗證失敗：{first_img}'
print('路徑驗證通過 ✓')

In [ ]:
# ── Step 4：產生 obj.names、obj.data、yolov4-tiny-custom.cfg ──
import re

list_dir   = os.path.join(DARKNET_DIR, 'data')
names_path = os.path.join(list_dir, 'obj.names')
data_path  = os.path.join(list_dir, 'obj.data')
cfg_path   = os.path.join(DARKNET_DIR, 'cfg', 'yolov4-tiny-custom.cfg')

# obj.names
with open(names_path, 'w') as f:
    f.write('\n'.join(CLASS_NAMES) + '\n')
print('obj.names 已建立 ✓')

# obj.data
has_test  = len(splits.get('test', [])) > 0
test_line = f'test  = {list_dir}/test.txt\n' if has_test else ''
with open(data_path, 'w') as f:
    f.write(
        f'classes = {NUM_CLASSES}\n'
        f'train   = {list_dir}/train.txt\n'
        f'valid   = {list_dir}/val.txt\n'
        f'{test_line}'
        f'names   = {names_path}\n'
        f'backup  = {BACKUP_DIR}\n'
    )
print('obj.data 已建立 ✓')

# yolov4-tiny-custom.cfg
!wget -q -O {cfg_path} \
    https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4-tiny-custom.cfg

with open(cfg_path, 'r') as f:
    cfg = f.read()

# ── 全域參數 ──────────────────────────────────────────────────
cfg = re.sub(r'\bbatch\s*=\s*\d+',       f'batch={BATCH_SIZE}',        cfg, count=1)
cfg = re.sub(r'\bsubdivisions\s*=\s*\d+',f'subdivisions={SUBDIVISIONS}',cfg, count=1)
cfg = re.sub(r'\bwidth\s*=\s*\d+',       f'width={INPUT_WIDTH}',        cfg, count=1)
cfg = re.sub(r'\bheight\s*=\s*\d+',      f'height={INPUT_HEIGHT}',      cfg, count=1)
cfg = re.sub(r'max_batches\s*=\s*\d+',   f'max_batches={MAX_BATCHES}',  cfg)
cfg = re.sub(r'steps\s*=\s*[\d,]+',      f'steps={STEPS1},{STEPS2}',    cfg)

# ── 修改兩個 [yolo] 層的 classes ──────────────────────────────
cfg = re.sub(r'\bclasses\s*=\s*\d+', f'classes={NUM_CLASSES}', cfg)

# ── 修改緊接在 [yolo] 前的 filters（最後一個 [convolutional] 的 filters）
# 找「filters=N」→「activation=linear」→（空行）→「[yolo]」這個模式
cfg = re.sub(
    r'(filters\s*=\s*)\d+(\s*\nactivation\s*=\s*linear\s*\n\n\[yolo\])',
    lambda m: f'{m.group(1)}{FILTERS}{m.group(2)}',
    cfg
)

with open(cfg_path, 'w') as f:
    f.write(cfg)

# 驗證替換結果
yolo_filters = re.findall(
    r'filters\s*=\s*(\d+)\s*\nactivation\s*=\s*linear\s*\n\n\[yolo\]', cfg
)
print('yolov4-tiny-custom.cfg 已建立 ✓')
print(f'  classes  = {NUM_CLASSES}（應有 2 個 [yolo] 層）')
print(f'  filters  = {yolo_filters}（兩個都應為 {FILTERS}）')
print(f'  max_batches={MAX_BATCHES}, steps={STEPS1},{STEPS2}')

if any(int(f) != FILTERS for f in yolo_filters) or len(yolo_filters) != 2:
    raise ValueError(
        f'cfg filters 替換失敗！預期兩個 [yolo] 前的 filters 均為 {FILTERS}，'
        f'實際得到 {yolo_filters}\n'
        '請手動檢查 cfg 檔案格式'
    )

In [ ]:
# ── Step 5：下載預訓練權重（Transfer Learning 用）──────────────
pretrained = os.path.join(DARKNET_DIR, 'yolov4-tiny.conv.29')
if not os.path.exists(pretrained):
    !wget -q --show-progress \
        https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29 \
        -O {pretrained}
print('預訓練權重已就緒 ✓')

In [ ]:
# ── Step 6：開始訓練 ───────────────────────────────────────────
import re as _re

LOG_FILE  = os.path.join(LOG_DIR, 'training_log.txt')
cfg_path  = os.path.join(DARKNET_DIR, 'cfg', 'yolov4-tiny-custom.cfg')
data_path = os.path.join(DARKNET_DIR, 'data', 'obj.data')
pretrained = os.path.join(DARKNET_DIR, 'yolov4-tiny.conv.29')

print('開始訓練，請耐心等待...')
print(f'訓練 log 儲存至：{LOG_FILE}')
print('─' * 60)

# -map 需要足夠的 val 圖片才穩定，圖片少時拿掉避免早退
map_flag = '-map' if len(splits.get('val', [])) >= 10 else ''

!{DARKNET_BIN} detector train \
    {data_path} \
    {cfg_path} \
    {pretrained} \
    -dont_show \
    {map_flag} \
    2>&1 | tee {LOG_FILE}

# ── 訓練後檢查 ────────────────────────────────────────────────
print('\n' + '─' * 60)
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        log_content = f.read()
    iterations = _re.findall(r'^\d+: ', log_content, _re.MULTILINE)
    if iterations:
        print(f'訓練完成 ✓  共執行 {len(iterations)} iterations')
    else:
        print('⚠ 偵測到 0 個 training iteration，訓練未正常執行')
        print('─── log 最後 30 行 ───')
        print('\n'.join(log_content.strip().split('\n')[-30:]))
        raise RuntimeError(
            '訓練異常結束。\n'
            '常見原因：\n'
            '  1. 圖片數量太少（目前 train 集 = '
            f'{len(splits["train"])} 張，建議每類至少 30 張）\n'
            '  2. BATCH_SIZE 大於 train 圖片數\n'
            '  3. .txt 標註格式有誤（class_id 超出範圍等）\n'
            '請查看上方 log 輸出確認錯誤原因'
        )
else:
    raise RuntimeError('找不到 log 檔，訓練未執行')

In [ ]:
# ── Step 7：確認輸出結果 ───────────────────────────────────────
import glob

best_w = os.path.join(BACKUP_DIR, 'yolov4-tiny-custom_best.weights')
last_w = os.path.join(BACKUP_DIR, 'yolov4-tiny-custom_last.weights')

print('=== 輸出結果 ===')
print(f'所有 weights 位於：{BACKUP_DIR}')
all_weights = sorted(glob.glob(os.path.join(BACKUP_DIR, '*.weights')))
for w in all_weights:
    size_mb = os.path.getsize(w) / 1024 / 1024
    tag = ' ← 推薦使用' if w == best_w else ''
    print(f'  {os.path.basename(w)}  ({size_mb:.1f} MB){tag}')

if not os.path.exists(best_w):
    print('⚠ best.weights 不存在（訓練 iteration 不足 1000 次，尚未觸發 mAP 評估）')

print(f'\n訓練 log    ：{LOG_FILE}')

In [ ]:
# ── Step 8（選用）：用 test set 評估 mAP ────────────────────────
best_w    = os.path.join(BACKUP_DIR, 'yolov4-tiny-custom_best.weights')
cfg_path  = os.path.join(DARKNET_DIR, 'cfg', 'yolov4-tiny-custom.cfg')
data_path = os.path.join(DARKNET_DIR, 'data', 'obj.data')

if splits.get('test') and os.path.exists(best_w):
    print('正在計算 test set mAP...')
    !{DARKNET_BIN} detector map {data_path} {cfg_path} {best_w} -points 0
elif not splits.get('test'):
    print('未設定 test set，跳過評估')
else:
    print('找不到 best.weights，跳過評估')

In [ ]:
# ── Step 9（選用）：繪製 loss 曲線 ─────────────────────────────
import matplotlib.pyplot as plt
import re

LOG_FILE = os.path.join(LOG_DIR, 'training_log.txt')

with open(LOG_FILE, 'r') as f:
    log = f.read()

pattern = re.compile(r'(\d+): ([\d.]+), ([\d.]+) avg loss')
iters, losses, avg_losses = [], [], []
for m in pattern.finditer(log):
    iters.append(int(m.group(1)))
    losses.append(float(m.group(2)))
    avg_losses.append(float(m.group(3)))

if iters:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(iters, losses,     alpha=0.4, linewidth=0.8, label='loss')
    ax.plot(iters, avg_losses, linewidth=1.5,            label='avg loss')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('YOLOv4-Tiny Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    chart_path = os.path.join(LOG_DIR, 'loss_curve.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Loss 曲線已儲存至：{chart_path}')
else:
    print('找不到 loss 記錄，請確認訓練已完成')